# Reddit Data Extraction
<!-- structured-notebook -->
## Notebook Summary
Purpose: decompress and merge the raw Reddit Pushshift archive (Zstandard-compressed `.zst` files, one per subreddit) into a single JSONL that later stages preprocess and model.

Main steps:
- Describe the data source (Academic Torrents distribution of Pushshift + post-April 2023 data by u/Watchful1).
- Walk through the decompression utilities that stream `.zst` files to JSON or CSV.
- Merge per-subreddit JSONL into a single `merged_submissions.jsonl` for use by `02_preprocessing_and_topic_modelling/`.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/input/raw/*.zst` | Pushshift archive (Academic Torrents) |
| Output | `<DATA_ROOT>/Reddit/output/merged_submissions.jsonl` | Consumed by `02_preprocessing_and_topic_modelling/01_preprocessing_and_topic_modelling.ipynb` |
| Output | `<DATA_ROOT>/Reddit/output/merged_submissions.csv` | Downstream EDA / external review |


# Data Source
<!-- structured-notebook -->
## Summary
The raw data comes from a publicly available Pushshift archive distributed via **Academic Torrents**. The archive combines pre-April 2023 Pushshift data with post-April 2023 data gathered by u/raiderbdev and repackaged by u/Watchful1.

Subreddits considered initially (9): `Aging`, `Biohackers`, `Health`, `HumanMicrobiome`, `longevity`, `Nootropics`, `ScientificNutrition`, `StackAdvice`, `Supplements`.

Final subreddits kept for modeling (3): `Biohackers`, `longevity`, `Aging`. The reduction was driven by thematic coherence — the other six subreddits introduced too much off-topic content relative to longevity narratives.

Raw files are Zstandard-compressed (`.zst`), one per subreddit, stored at `SocialMedia/Reddit/data/raw/`. They are not committed to this repo because of their size; obtain them from Academic Torrents or coordinate with the project maintainer for access.


# Utility: Single-File `.zst` → CSV
<!-- structured-notebook -->
## Summary
The simplest extraction path: decompress one `.zst` file and flatten each submission to a CSV row with a few chosen columns. Good for quick inspection and small samples. The code block below is the canonical `to_csv.py` script. Edit the hardcoded `input_file` path at the top to point at your `.zst`.


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
# this converts a zst file to csv
#
# it's important to note that the resulting file will likely be quite large
# and you probably won't be able to open it in excel or another csv reader
#
# arguments are inputfile, outputfile, fields
# call this like
# python to_csv.py wallstreetbets_submissions.zst wallstreetbets_submissions.csv author,selftext,title

import zstandard
import os
import json
import sys
import csv
from datetime import datetime
import logging.handlers


# put the path to the input file
input_file_path = rREDDIT_OUTPUT / "Biohackers_submissions.zst"
# put the path to the output file, with the csv extension
output_file_path = rREDDIT_OUTPUT / "Biohackers_submissions.csv"
# if you want a custom set of fields, put them in the following list. If you leave it empty the script will use a default set of fields
fields = []

log = logging.getLogger("bot")
log.setLevel(logging.DEBUG)
log.addHandler(logging.StreamHandler())


def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
	chunk = reader.read(chunk_size)
	bytes_read += chunk_size
	if previous_chunk is not None:
		chunk = previous_chunk + chunk
	try:
		return chunk.decode()
	except UnicodeDecodeError:
		if bytes_read > max_window_size:
			raise UnicodeError(f"Unable to decode frame after reading {bytes_read:,} bytes")
		return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)


def read_lines_zst(file_name):
	with open(file_name, 'rb') as file_handle:
		buffer = ''
		reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_handle)
		while True:
			chunk = read_and_decode(reader, 2**27, (2**29) * 2)
			if not chunk:
				break
			lines = (buffer + chunk).split("\n")

			for line in lines[:-1]:
				yield line, file_handle.tell()

			buffer = lines[-1]
		reader.close()


if __name__ == "__main__":
	if len(sys.argv) >= 3:
		input_file_path = sys.argv[1]
		output_file_path = sys.argv[2]
		#fields = sys.argv[3].split(",")

	is_submission = "submission" in input_file_path
	if not len(fields):
		if is_submission:
			fields = ["author","title","score","created","link","text","url"]
		else:
			fields = ["author","score","created","link","body"]

	file_size = os.stat(input_file_path).st_size
	file_lines, bad_lines = 0, 0
	line, created = None, None
	output_file = open(output_file_path, "w", encoding='utf-8', newline="")
	writer = csv.writer(output_file)
	writer.writerow(fields)
	try:
		for line, file_bytes_processed in read_lines_zst(input_file_path):
			try:
				obj = json.loads(line)
				output_obj = []
				for field in fields:
					if field == "created":
						value = datetime.fromtimestamp(int(obj['created_utc'])).strftime("%Y-%m-%d %H:%M")
					elif field == "link":
						if 'permalink' in obj:
							value = f"https://www.reddit.com{obj['permalink']}"
						else:
							value = f"https://www.reddit.com/r/{obj['subreddit']}/comments/{obj['link_id'][3:]}/_/{obj['id']}/"
					elif field == "author":
						value = f"u/{obj['author']}"
					elif field == "text":
						if 'selftext' in obj:
							value = obj['selftext']#[:32000] # remove first # if the subreddit has very large text posts and you want to open this in excel
						else:
							value = ""
					else:
						value = obj[field]

					output_obj.append(str(value).encode("utf-8", errors='replace').decode())
				writer.writerow(output_obj)

				created = datetime.utcfromtimestamp(int(obj['created_utc']))
			except json.JSONDecodeError as err:
				bad_lines += 1
			file_lines += 1
			if file_lines % 100000 == 0:
				log.info(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {(file_bytes_processed / file_size) * 100:.0f}%")
	except KeyError as err:
		log.info(f"Object has no key: {err}")
		log.info(line)
	except Exception as err:
		log.info(err)
		log.info(line)

	output_file.close()
	log.info(f"Complete : {file_lines:,} : {bad_lines:,}")

# Utility: Batch Decompress a Directory
<!-- structured-notebook -->
## Summary
Batch-decompress every `.zst` in a directory into JSONL (one JSON object per line). Preserves all raw fields; downstream preprocessing filters to the columns it actually needs. Use this over the CSV path when you want full metadata retained.


In [ ]:
import os
import sys
import zstandard
import logging
import json
from datetime import datetime

log = logging.getLogger("bot")
log.setLevel(logging.DEBUG)
log.addHandler(logging.StreamHandler())

# ← Your existing functions unchanged →

def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
    chunk = reader.read(chunk_size)
    bytes_read += chunk_size
    if previous_chunk is not None:
        chunk = previous_chunk + chunk
    try:
        return chunk.decode()
    except UnicodeDecodeError:
        if bytes_read > max_window_size:
            raise UnicodeError(f"Unable to decode after {bytes_read:,} bytes")
        log.info(f"Decoding error at {bytes_read:,} bytes; reading more…")
        return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)

def read_lines_zst(file_obj):
    buffer = ''
    reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_obj)
    while True:
        chunk = read_and_decode(reader, 2**27, (2**29) * 2)
        if not chunk:
            break
        lines = (buffer + chunk).split("\n")
        for line in lines[:-1]:
            yield line
        buffer = lines[-1]
    reader.close()

def process_and_decompress(in_path, out_path):
    """Decompress and write to output, plus process JSON lines."""
    total_lines = bad_lines = 0
    size = os.stat(in_path).st_size
    file_bytes_processed = 0
    created = None

    with open(in_path, 'rb') as ifh, open(out_path, 'w', encoding='utf-8') as ofh:
        for line in read_lines_zst(ifh):
            ofh.write(line + "\n")
            file_bytes_processed = ifh.tell()
            total_lines += 1
            try:
                obj = json.loads(line)
                created = datetime.utcfromtimestamp(int(obj['created_utc']))
            except (KeyError, json.JSONDecodeError):
                bad_lines += 1
            if total_lines % 100_000 == 0:
                log.info(f"{created:%Y-%m-%d %H:%M:%S} • lines={total_lines:,} • bad={bad_lines:,} • bytes={file_bytes_processed:,} ({file_bytes_processed/size:.0%})")

    log.info(f"Done {in_path}: processed {total_lines:,} lines, {bad_lines:,} malformed")

def main(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, _, files in os.walk(input_dir):
        for name in files:
            if name.endswith(".zst"):
                in_file = os.path.join(root, name)
                out_name = os.path.splitext(name)[0] + ".jsonl"
                out_file = os.path.join(output_dir, out_name)
                log.info(f"→ {in_file} → {out_file}")
                process_and_decompress(in_file, out_file)

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <input_folder> <output_folder>")
        sys.exit(1)
    main(sys.argv[1], sys.argv[2])

# Utility: Merge per-Subreddit JSONL
<!-- structured-notebook -->
## Summary
After decompression, each subreddit has its own JSONL file. This step concatenates them into a single `merged_submissions.jsonl` (~2.6 GB, ~79,152 posts across the three retained subreddits) which becomes the input for `02_preprocessing_and_topic_modelling/`.

The merge is a straight line-by-line append — no deduplication, no filtering. Dedup happens later in the preprocessing stage so that the mapping between raw posts and unique preprocessed strings is preserved and can be reconstructed.


In [ ]:
import os

folder_path = REDDIT_OUTPUT
output_file = os.path.join(folder_path, "merged_submissions.jsonl")

with open(output_file, "w", encoding="utf-8") as outfile:
    for filename in os.listdir(folder_path):
        if "_submissions" in filename and filename.endswith(".jsonl") and filename != "merged_submissions.jsonl":
            file_path = os.path.join(folder_path, filename)
            print(f"Merging: {filename}")
            with open(file_path, "r", encoding="utf-8") as infile:
                for line in infile:
                    outfile.write(line)

print("All files merged successfully.")

# Utility: Sample and Inspect a `.zst`
<!-- structured-notebook -->
## Summary
Lightweight sampling utility to peek into a compressed dump without fully decompressing it. Useful for sanity-checking field availability (score, num_comments, created_utc, selftext, permalink, etc.) before committing to a full pipeline run.


In [ ]:
import os
import pickle
import json
import pandas as pd
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# -------------------------------
# 1) RAW FILE PREVIEW + COLUMNS
# -------------------------------
def preview_raw_file(raw_path: str, nrows: int = 5) -> pd.DataFrame:
    """Load a small preview of the raw file and print columns."""
    raw_path = str(raw_path)
    if raw_path.endswith(".csv"):
        df = pd.read_csv(raw_path, nrows=nrows)
    elif raw_path.endswith(".parquet"):
        df = pd.read_parquet(raw_path).head(nrows)
    elif raw_path.endswith(".json") or raw_path.endswith(".jsonl"):
        # pandas supports nrows for JSON lines too
        df = pd.read_json(raw_path, lines=True, nrows=nrows)
    else:
        raise ValueError("Unsupported file format. Use CSV, Parquet, JSON, or JSONL.")

    print("\n=== RAW FILE PREVIEW ===")
    print(df.head(nrows))
    print("\n=== RAW FILE COLUMNS ===")
    print(list(df.columns))
    return df


# -------------------------------
# 2) GENERIC PICKLE INSPECTOR
# -------------------------------
def inspect_pickle(pkl_path: str, max_preview: int = 2):
    """Print basic info + small preview for a pickle file."""
    p = Path(pkl_path)
    if not p.exists():
        print(f"File not found: {pkl_path}")
        return None

    print(f"\n=== Inspecting {pkl_path} ===")
    try:
        with open(pkl_path, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        print(f"Could not load {pkl_path}: {e}")
        return None

    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"Type: dict | Keys: {len(keys)} | Example keys: {keys[:5]}")
        first_key = keys[0] if keys else None
        if first_key is not None:
            print(f"Sample value for key={first_key}:")
            val = obj[first_key]
            if isinstance(val, list):
                print(val[:max_preview])
            else:
                print(str(val)[:500])
    elif isinstance(obj, list):
        print(f"Type: list | Length: {len(obj)} | First items: {obj[:max_preview]}")
    else:
        print(f"Type: {type(obj)}")
        print(str(obj)[:500])

    return obj

inspect_pickle("../../main_topic_models/bertopic_all-mpnet-base-v2_final_output/topic_doc_map.pkl", 2)
inspect_pickle("../../preprocessed_docs/preprocessed_reddit_docs.pkl", 2)

# Output
<!-- structured-notebook -->
## Summary
After running this notebook's utilities in order, the canonical artifact is `SocialMedia/Reddit/data/raw/merged_submissions.jsonl`. Record counts per step:

- Raw submissions across 3 subreddits: 79,152
- After language + length + preprocessing filter (in stage 02): 75,049 (94.8%)
- After deduplication (in stage 02): 72,806 unique posts

The next stage is `02_preprocessing_and_topic_modelling/01_preprocessing_and_topic_modelling.ipynb`.


---
<!-- nav-footer -->

[Project Overview](../../../../../PROJECT_OVERVIEW.ipynb) | [01 preprocessing and topic modelling](../../../../../SocialMedia/Reddit/notebooks/BERTopic/02_preprocessing_and_topic_modelling/01_preprocessing_and_topic_modelling.ipynb) →
